# The Second-Order Problem
## Why do new users not come back — and which fix is worth the most GMV?

**Product-analytics case study on a food-delivery marketplace · Jan–Jun 2026 (simulated data)**

---

### How to read this notebook

Every section follows the same shape:

> **Question → SQL → result → *so what?***

The SQL lives in `sql/*.sql` (not inline), because in a real team the query *is* the
artifact other people reuse — the notebook is just the narrative wrapped around it.
`sqlkit` loads named query blocks out of those files.

**The one thing to take away:** this project is judged on the *decision*, not the code.
Each analysis below ends with the business implication, and the whole thing converges on
one recommendation with a rupee value attached.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath("../src"))
import pandas as pd
from sqlkit import connect, resolve

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 30)

a = connect()          # opens data/zomato.db and parses every sql/*.sql block
print("SQL blocks available:", ", ".join(sorted(a.blocks)))

SQL blocks available: ab_balance, ab_by_segment, ab_guardrails, ab_primary, behaviour_by_channel, cohort_by_late, cohort_matrix, dose_response, first_order_features, funnel_by_city, funnel_by_platform, funnel_overall, guardrails, headline, hour_of_day, late_by_city_hour, monthly_trend, north_star, payment_leak_size, repeat_by_late, repeat_by_late_stratified, repeat_by_segment, rfm_summary, rfm_users, srm_check


---
## 0. The data

Four tables, shaped the way a real event-driven product warehouse is shaped:

| table | grain | what it is |
|---|---|---|
| `users` | one row per user | signup date, city, platform, acquisition channel, Gold flag |
| `orders` | one row per order | GMV, discount, promised vs actual delivery time, rating, status |
| `app_events` | one row per session × funnel step | `app_open → search → restaurant_view → add_to_cart → checkout_start → payment_success` |
| `ab_test_assignments` | one row per experiment participant | variant, primary metric, guardrail metrics |

**The data is simulated, on purpose and transparently.** Public food-delivery datasets
have orders but no clickstream and no experiment assignments, which makes funnel and A/B
analysis impossible. `src/generate_data.py` documents the causal structure it injects;
the job of this notebook is to *recover* that structure using the same techniques you
would use in production.

In [2]:
for t in ["users", "orders", "app_events", "ab_test_assignments"]:
    n = a.raw(f"SELECT COUNT(*) AS n FROM {t}").iloc[0, 0]
    print(f"{t:<22} {n:>10,} rows")

a.raw("SELECT * FROM orders LIMIT 5")

users                      60,000 rows
orders                     71,599 rows
app_events              1,242,383 rows
ab_test_assignments        19,301 rows


,order_id,user_id,order_ts,order_seq,city,platform,cuisine,gmv,discount,delivery_fee,promised_minutes,delivery_minutes,is_late,rating,status
0,1,1,2026-06-04 21:16:00,1,Pune,Android,South Indian,588.0,135.0,35,32.0,40.5,0,4.5,delivered
1,2,2,2026-04-03 17:35:00,1,Hyderabad,iOS,Burgers,502.0,158.0,0,32.0,23.2,0,5.0,delivered
2,3,2,2026-04-12 12:17:00,2,Hyderabad,iOS,North Indian,579.0,25.0,25,32.0,30.6,0,4.0,delivered
3,4,2,2026-04-13 12:54:00,3,Hyderabad,iOS,Biryani,445.0,63.0,45,32.0,31.3,0,5.0,delivered
4,5,3,2026-06-10 12:12:00,1,Pune,iOS,South Indian,328.0,101.0,25,32.0,26.6,0,4.0,delivered


---
## 1. Metric definition — before looking at anything

Doing this first is the discipline that separates an analyst from a chart generator.
If you define the metric *after* seeing the data, you will define whichever metric looks
best.

**North star — 30-day repeat rate of new users.**
Of users whose first order was ≥30 days before the data cut-off, what share ordered again
within 30 days?

Why this one:
- In food delivery, **the second order is the strongest early predictor of LTV** — most
  of the churn happens between order 1 and order 2.
- It is **fast-moving**: you see the effect of a change within weeks, unlike LTV.
- It **cannot be gamed by spending more on acquisition**, unlike total orders or GMV. A
  metric that goes up when you buy traffic is a vanity metric.
- It is restricted to **matured cohorts** so right-censoring does not flatter it — a user
  who ordered yesterday hasn't had 30 days to come back and must not be counted as churned.

**Guardrails** (must not get worse while we chase the north star): cancellation rate,
average delivery time, average rating, discount % of GMV, net revenue per order.

In [3]:
print(resolve(a.blocks, "north_star"))

-- NORTH STAR: 30-day repeat rate of new users
--   = of all users whose FIRST order was >=30 days before the data cut-off,
--     what share placed a SECOND order within 30 days of the first?
-- Why this metric: it is the earliest reliable predictor of lifetime value in food
-- delivery, it is fast to move, and unlike "orders" it cannot be faked by buying
-- more traffic. Restricting to fully-matured cohorts avoids right-censoring bias.
WITH firsts AS (
    SELECT user_id,
           MIN(order_ts) AS first_ts
    FROM orders
    WHERE status = 'delivered'
    GROUP BY user_id
),
matured AS (              -- only users with a full 30-day window of observation
    SELECT f.user_id, f.first_ts
    FROM firsts f
    WHERE JULIANDAY('2026-06-30') - JULIANDAY(f.first_ts) >= 30
),
second AS (
    SELECT m.user_id,
           MAX(CASE WHEN o.order_ts > m.first_ts
                     AND JULIANDAY(o.order_ts) - JULIANDAY(m.first_ts) <= 30
                    THEN 1 ELSE 0 END) AS repeated_30d

In [4]:
headline = a.q("headline")
north_star = a.q("north_star")
display(headline.T.rename(columns={0: "value"}))
display(north_star)

,value
registered_users,60000.00
ordering_users,40389.00
activation_rate_pct,67.30
orders,70005.00
gmv,31277157.00
aov,447.00
orders_per_ordering_user,1.73
late_delivery_rate_pct,20.70
avg_rating,4.23


,matured_new_users,repeated_users,repeat_30d_rate_pct
0,31727,9503,29.95


### So what?

**67.3% of registered users place a first order — but only 29.9% of them place a
second one within 30 days.** Acquisition is working; retention is not. Every rupee of
acquisition spend is being amortised over 1.73 orders.

That reframes the whole project: the problem is not the top of the funnel, it is **the
second order**. Everything below is an attempt to find out where that second order goes.

---
## 2. Cohort retention — is it getting better or worse?

A cohort is all users whose **first order** fell in month *M*. Retention in period *k* is
the share of that cohort that ordered at all in month *M+k*.

Reading a cohort table has a fixed grammar:
- **Down a column** = are newer cohorts better or worse than older ones? (product/quality trend)
- **Across a row** = how fast does a single cohort decay? (stickiness)

In [5]:
cm = a.q("cohort_matrix")
piv = cm.pivot(index="cohort_month", columns="period_index", values="retention_pct")
piv.style.background_gradient(cmap="Reds", axis=None).format("{:.0f}", na_rep="")

period_index,0,1,2,3,4,5
cohort_month,,,,,,
2026-01,100,27,18,11,6,3
2026-02,100,27,16,9,5,
2026-03,100,24,11,5,,
2026-04,100,23,7,,,
2026-05,100,19,,,,
2026-06,100,,,,,


### So what?

Two things, and the second is the alarming one:

1. **Across a row:** ~73% of the January cohort never returned in month 1. The decay is
   brutal and consistent across cohorts.
2. **Down the M1 column:** 27% → 27% → 24% → 23% → 19%. **Newer cohorts are worse.** We
   are growing order volume while the quality of each new cohort deteriorates.

*Caveat I have to state:* the May cohort has had less calendar time inside the window, so
some of that decline is right-censoring, not decay. That is why the north-star metric is
computed only on matured cohorts — but the direction is consistent enough to investigate.

So: what changed for newer cohorts?

---
## 3. The hypothesis: the first delivery *is* the product

For a first-time user, the app is not the product — **the first delivery is**. If it
arrives late, the user has learned that the promise is unreliable.

Let's split the retention curve by whether the user's very first order arrived more than
10 minutes past the promised ETA.

In [6]:
cbl = a.q("cohort_by_late")
cbl.pivot(index="period_index", columns="segment", values="retention_pct")

segment,First delivery LATE,First delivery ON TIME
period_index,,
0,100.0,100.0
1,12.9,20.1
2,5.5,8.0
3,2.4,3.7
4,1.0,1.5
5,0.3,0.4


In [7]:
display(a.q("repeat_by_late"))
display(a.q("dose_response"))

,first_delivery,users,repeat_30d_pct
0,Late,6999,20.47
1,On time,24728,32.64


,lateness_bucket,users,avg_first_rating,repeat_30d_pct
0,1. Early / on time,7196,4.52,32.91
1,2. 0-10 min late,17592,4.27,32.53
2,3. 10-20 min late,6376,3.82,20.36
3,4. 20-30 min late,536,3.28,20.90
4,5. 30+ min late,27,2.89,11.11


### So what?

- Late-first-delivery users repeat at **20.5%** vs **32.6%** — a **12.2pp** gap, and the
  whole retention curve sits lower for months afterwards. One bad delivery is not a
  one-off cost; it is a permanent haircut on that user's lifetime value.
- The **dose–response** is the important part: nothing happens for the first 10 minutes
  (users forgive a small overrun), then repeat rate falls off a cliff. That shape is very
  hard to produce by confounding, and it is why the 10-minute threshold is the right
  definition of "late".

But correlation is not causation. Maybe late orders happen in Bengaluru, at peak, to
paid-social users — and *those* users would have churned anyway.

---
## 4. Killing the confounders

Three tests before I believe this:

1. **Stratification** — compare late vs on-time users *within* the same city × channel ×
   Gold × platform cell, then re-weight by cell size. If the gap survives, it isn't
   composition.
2. **Dose–response** — done above.
3. **A randomised experiment** — section 7.

In [8]:
display(a.q("repeat_by_late_stratified"))
display(a.q("repeat_by_segment"))

,cells_used,users_covered,adjusted_gap_pp,wtd_ontime_pct,wtd_late_pct
0,69,30054,11.78,31.65,19.87


,dimension,value,users,repeat_pct
0,channel,referral,4587,35.49
1,channel,organic,11277,33.59
2,channel,paid_search,7290,30.66
3,channel,paid_social,8573,21.60
4,city,Hyderabad,4355,31.30
5,city,Mumbai,5783,30.75
6,city,Kolkata,2908,30.57
7,city,Pune,3424,30.49
8,city,Delhi NCR,6898,29.62
9,city,Bengaluru,8359,28.54


### So what?

Within 69 like-for-like cells covering 30,054 users, the gap only narrows from
**12.2pp to 11.8pp**. Composition explains almost none of it.

The segment table also ranks every other candidate driver, which is how I can claim
lateness is *the biggest* one rather than just *a* one:

| driver | spread in 30-day repeat rate |
|---|---|
| **Late first delivery** | **12.2pp** |
| Gold membership | ~22pp — but that is selection: people who subscribe already intended to order more |
| Acquisition channel | ~14pp (referral vs paid-social) |
| First-order rating | ~12pp — largely *downstream of* lateness, not independent of it |
| City | ~3pp |
| Platform | ~1.4pp |

Gold and channel are things we *select*, not things we *do*. Lateness is something we
**control** — which makes it the actionable driver.

---
## 5. The funnel — the other place orders leak

Retention is about users who *ordered*. The funnel is about sessions that *tried to*.
A session counts at a step if it fired that event at least once.

In [9]:
display(a.q("funnel_overall"))
display(a.q("funnel_by_platform"))
display(a.q("payment_leak_size"))

,step_no,event_name,sessions,pct_of_top,step_conv_pct,step_dropoff_pct
0,1,app_open,342573,100.0,NaN,NaN
1,2,search,295276,86.2,86.2,13.8
2,3,restaurant_view,246474,71.9,83.5,16.5
3,4,add_to_cart,179640,52.4,72.9,27.1
4,5,checkout_start,107128,31.3,59.6,40.4
5,6,payment_success,71292,20.8,66.5,33.5


,platform,step_no,event_name,sessions,step_conv_pct
0,Android,1,app_open,260952,NaN
1,Android,2,search,224916,86.2
2,Android,3,restaurant_view,187802,83.5
3,Android,4,add_to_cart,136879,72.9
4,Android,5,checkout_start,81601,59.6
5,Android,6,payment_success,50613,62.0
6,iOS,1,app_open,81621,NaN
7,iOS,2,search,70360,86.2
8,iOS,3,restaurant_view,58672,83.4
9,iOS,4,add_to_cart,42761,72.9


,android_pay_conv_pct,ios_pay_conv_pct,recoverable_orders_6mo
0,62.0,81.0,15491.0


### So what?

The overall funnel says the biggest single drop is **checkout → payment (34%)**. That
alone is a weak finding — carts get abandoned everywhere.

Segmenting it is what turns it into an insight: **Android converts at 62.0% on that
step, iOS at 81.0%**, while *every earlier step matches within 0.2pp*. A gap isolated to
one step on one platform is the signature of a **defect** (payment SDK / UPI intent
failure), not of different user intent. That is an engineering ticket, not a growth
experiment — and it is worth ~15,491 orders over six months.

**This is the single most valuable habit in funnel analysis: never read an aggregate
funnel without segmenting it.**

---
## 6. Who are our users? RFM segmentation

RFM = Recency, Frequency, Monetary. Score each user 1–5 on each (via `NTILE(5)`), then
map score combinations to segments a PM can actually act on.

In [10]:
display(a.q("rfm_summary"))
display(a.q("behaviour_by_channel"))

,segment,users,pct_users,gmv_lakh,pct_gmv,avg_orders,avg_lifetime_gmv,avg_recency_days
0,Champions,7551,18.7,115.4,36.9,3.03,1529.0,20.0
1,Loyal,7929,19.6,68.4,21.9,1.91,863.0,46.0
2,At Risk (was valuable),5100,12.6,53.6,17.1,2.39,1051.0,116.0
3,Hibernating / Churned,7403,18.3,27.8,8.9,1.00,376.0,125.0
4,Needs Attention,6697,16.6,25.4,8.1,1.00,379.0,97.0
5,New / Promising,5709,14.1,22.1,7.1,1.00,387.0,20.0


,acquisition_channel,ordering_users,repeat_rate_pct,orders_per_user,gmv_per_user,discount_pct_of_gmv
0,referral,5858,32.0,1.86,848.0,18.0
1,organic,14343,30.1,1.85,851.0,18.0
2,paid_search,9292,27.9,1.75,790.0,18.6
3,paid_social,10896,19.4,1.49,620.0,26.5


### So what?

- **Champions (~19% of users) carry ~36% of GMV.** Retention spend should be weighted
  toward keeping them, not toward blanket discounts.
- **Paid-social is our worst channel on every axis**: lowest repeat rate, lowest GMV per
  user, highest discount burn. We are buying discount-chasers. Referral is the opposite.
- The correct read of the "Hibernating / Churned" bucket is that most of them are
  **one-and-done users** — the same second-order problem, seen from a different angle.

---
## 7. The experiment: "Next-Order Nudge"

Everything so far is observational. Here is the randomised evidence.

> **Hypothesis:** sending a ₹75 coupon valid 7 days immediately after a user's first
> delivered order increases the share who order again within 14 days.
>
> - **Unit of randomisation:** user, assigned at first-order completion
> - **Split:** 50/50 · **Primary metric:** repeat order within 14 days
> - **MDE:** 2.0pp absolute, 80% power, α = 0.05 (two-sided)
> - **Guardrails:** second-order AOV, second-order cancellation rate, coupon cost

The order of operations matters and most people get it wrong: **sample size is computed
before the test runs, and SRM is checked before the result is read.**

In [11]:
from run_analysis import sample_size_per_arm, ab_readout

prim = a.q("ab_primary")
c = prim[prim.variant == "control"].iloc[0]
t = prim[prim.variant == "treatment"].iloc[0]

need = sample_size_per_arm(p_base=c.repeat_14d_pct / 100, mde_abs=0.02)
print(f"Required users per arm for a 2.0pp MDE at 80% power: {need:,}")
print(f"Actual: control {int(c.n):,} / treatment {int(t.n):,} -> "
      f"{'adequately powered' if min(c.n, t.n) >= need else 'UNDERPOWERED'}")

display(a.q("srm_check"))     # sample ratio mismatch: must be ~50/50
display(a.q("ab_balance"))    # pre-assignment covariates must match
display(prim)

Required users per arm for a 2.0pp MDE at 80% power: 6,275
Actual: control 9,681 / treatment 9,620 -> adequately powered


,variant,users,pct
0,control,9681,50.16
1,treatment,9620,49.84


,variant,users,first_order_aov,first_late_pct,first_rating
0,control,9681,396.9,22.26,4.214
1,treatment,9620,399.2,21.92,4.213


,variant,n,conversions,repeat_14d_pct
0,control,9681,1838,18.986
1,treatment,9620,2143,22.277


In [12]:
res = ab_readout(int(t.conversions), int(t.n), int(c.conversions), int(c.n))
pd.Series(res).to_frame("value")

,value
control_n,9681
treatment_n,9620
control_rate_pct,18.986
treatment_rate_pct,22.277
se_control_pct,0.399
se_treatment_pct,0.424
abs_lift_pp,3.291
rel_lift_pct,17.33
ci_low_pp,2.15
ci_high_pp,4.432


In [13]:
display(a.q("ab_guardrails"))
display(a.q("ab_by_segment"))

,variant,n,second_order_aov,second_cancel_pct,total_coupon_cost
0,control,9681,431.2,2.56,0.0
1,treatment,9620,404.9,1.87,160725.0


,segment,variant,n,conversions,repeat_14d_pct
0,First delivery LATE,control,2155,282,13.09
1,First delivery LATE,treatment,2109,319,15.13
2,First delivery ON TIME,control,7526,1556,20.67
3,First delivery ON TIME,treatment,7511,1824,24.28


### So what? — the ship / don't-ship decision

**The lift is real:** +3.29pp absolute (+17.3% relative), 95% CI
[2.15pp, 4.43pp], p < 0.0001. SRM passes and the arms are balanced on
pre-assignment covariates, so the randomisation is trustworthy.

**But the guardrail moved:** treated users' second-order AOV is **₹26 lower** — they
trade down to make the coupon worth using. Cancellation rate is unchanged. Netting the
coupon cost *and* the AOV loss against the incremental orders, the nudge is still
positive, but it is worth ~₹11L/yr rather than the ~₹25L/yr the gross lift suggests.

**And the effect is not where I hoped:** the lift is +3.61pp for users whose first
delivery went well and only +2.04pp for those whose went badly. **The coupon buys an extra
order; it does not repair a broken experience.** So it is not a substitute for fixing
delivery — which is the finding a growth team is most likely to talk itself out of.

**Decision: ship it, targeted, with a permanent 10% holdout.** Blanket-sending it pays
users who would have come back anyway.

---
## 8. Sizing it in rupees

An analysis that stops at "significant" is unfinished. Every recommendation gets a number
and a stated assumption, so a sceptic can attack the assumption rather than the vibe.

In [14]:
import json
imp = json.load(open("../reports/results.json"))["impact"]
pd.Series(imp)[["android_fix_gmv_lakh_yr", "lateness_fix_gmv_lakh_yr",
                "nudge_gross_gmv_lakh_yr", "nudge_coupon_cost_lakh_yr",
                "nudge_aov_guardrail_cost_lakh_yr", "nudge_net_gmv_lakh_yr",
                "total_gmv_lakh_yr"]].to_frame("₹ lakh / year")

,₹ lakh / year
android_fix_gmv_lakh_yr,69.2
lateness_fix_gmv_lakh_yr,5.5
nudge_gross_gmv_lakh_yr,25.4
nudge_coupon_cost_lakh_yr,10.6
nudge_aov_guardrail_cost_lakh_yr,3.7
nudge_net_gmv_lakh_yr,11.1
total_gmv_lakh_yr,85.8


**Assumptions, stated plainly:**

| assumption | value | why it is conservative |
|---|---|---|
| Android payment fix closes | **half** the iOS gap | full parity after one release is optimistic |
| Late-delivery rate reduction | **6pp** in the worst cells | achievable with rider capacity + honest peak ETAs |
| Effect of lateness | the **stratified 11.8pp**, not the raw 12.2pp | strips out composition |
| Value of a retained user | **2.72 extra orders** (measured) | right-censored, so an under-estimate |
| Nudge cost | coupon **and** the AOV guardrail loss | most write-ups count only the coupon |

**Total ≈ ₹86 lakh / year on a 60k-user base.**

---
## 9. What I would do next

1. **Run the geo experiment on ETAs.** The lateness finding is the biggest one and the
   least clean. Add rider capacity in two matched cities at peak and measure new-user
   repeat rate. That converts a 11.8pp observational gap into a causal one.
2. **Extend the horizon to 90 days.** If the nudge only pulls an order forward, its
   14-day win is an illusion. The permanent holdout answers this.
3. **Rebalance acquisition spend** from paid-social toward referral, and re-measure
   blended CAC payback per channel rather than CAC alone.
4. **Keep it alive:** `src/monitor.py` re-runs these metrics weekly against the same SQL,
   flags RED/AMBER breaches, and exits non-zero so a scheduler can alert on it.

In [15]:
a.close()